In [ ]:
!unzip /content/Original.zip

Archive:  /content/Original.zip
  inflating: Original/Beads_002_jpg.rf.4fbace3e0639d30f5ad863f056da749d.jpg  
  inflating: Original/Beads_002_jpg.rf.51bd1ed2b14a4da0ae5e63197abffcfe.jpg  
  inflating: Original/Beads_002_jpg.rf.fd056108dd8da331747ee5bc5518e30b.jpg  
  inflating: Original/Beads_003_jpg.rf.2b0e56217c4eff05328444469f73de22.jpg  
  inflating: Original/Beads_003_jpg.rf.74ab1bae8fb635a5e001b1c75e11550b.jpg  
  inflating: Original/Beads_003_jpg.rf.ee32e66b033b66b1fc9ff1247ba29a82.jpg  
  inflating: Original/Beads_004_jpg.rf.350cb6f53dfe1bc289353841d48928b0.jpg  
  inflating: Original/Beads_004_jpg.rf.9bd163eb8c22a189036df34bbaef9ad4.jpg  
  inflating: Original/Beads_004_jpg.rf.ed2a04ffc7284393fe915d90ed89295b.jpg  
  inflating: Original/Beads_005_jpg.rf.53e1ba3e86e0e7bda3f33a9aa03f4a55.jpg  
  inflating: Original/Beads_005_jpg.rf.7b7deefc1c2693ef6bbb240f178b231b.jpg  
  inflating: Original/Beads_005_jpg.rf.e28c8fd8affe57898ff0730ceff5039c.jpg  
  inflating: Original/Beads_007_

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from skimage import morphology, measure
import urllib.request
import gdown

# Basic building block for U²-Net
class REBNCONV(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, dirate=1):
        super(REBNCONV, self).__init__()
        self.conv_s1 = nn.Conv2d(in_ch, out_ch, 3, padding=1*dirate, dilation=1*dirate)
        self.bn_s1 = nn.BatchNorm2d(out_ch)
        self.relu_s1 = nn.ReLU(inplace=True)

    def forward(self, x):
        hx = x
        xout = self.relu_s1(self.bn_s1(self.conv_s1(hx)))
        return xout

# RSU7 Module (Fixed)
class RSU7(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(RSU7, self).__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch, dirate=1)
        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool3 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool4 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv5 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool5 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv6 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.rebnconv7 = REBNCONV(mid_ch, mid_ch, dirate=2)
        self.rebnconv6d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv5d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv4d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv3d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv2d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv1d = REBNCONV(mid_ch*2, out_ch, dirate=1)  # Changed to output out_ch

    def forward(self, x):
        hx = x
        hxin = self.rebnconvin(hx)
        hx1 = self.rebnconv1(hxin)
        hx = self.pool1(hx1)
        hx2 = self.rebnconv2(hx)
        hx = self.pool2(hx2)
        hx3 = self.rebnconv3(hx)
        hx = self.pool3(hx3)
        hx4 = self.rebnconv4(hx)
        hx = self.pool4(hx4)
        hx5 = self.rebnconv5(hx)
        hx = self.pool5(hx5)
        hx6 = self.rebnconv6(hx)
        hx7 = self.rebnconv7(hx6)
        hx6d = self.rebnconv6d(torch.cat((hx7, hx6), 1))
        hx6dup = _upsample_like(hx6d, hx5)
        hx5d = self.rebnconv5d(torch.cat((hx6dup, hx5), 1))
        hx5dup = _upsample_like(hx5d, hx4)
        hx4d = self.rebnconv4d(torch.cat((hx5dup, hx4), 1))
        hx4dup = _upsample_like(hx4d, hx3)
        hx3d = self.rebnconv3d(torch.cat((hx4dup, hx3), 1))
        hx3dup = _upsample_like(hx3d, hx2)
        hx2d = self.rebnconv2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)
        hx1d = self.rebnconv1d(torch.cat((hx2dup, hx1), 1))
        return hx1d + hxin

# RSU6 Module (Fixed)
class RSU6(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(RSU6, self).__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch, dirate=1)
        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool3 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool4 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv5 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.rebnconv6 = REBNCONV(mid_ch, mid_ch, dirate=2)
        self.rebnconv5d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv4d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv3d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv2d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv1d = REBNCONV(mid_ch*2, out_ch, dirate=1)  # Changed to output out_ch

    def forward(self, x):
        hx = x
        hxin = self.rebnconvin(hx)
        hx1 = self.rebnconv1(hxin)
        hx = self.pool1(hx1)
        hx2 = self.rebnconv2(hx)
        hx = self.pool2(hx2)
        hx3 = self.rebnconv3(hx)
        hx = self.pool3(hx3)
        hx4 = self.rebnconv4(hx)
        hx = self.pool4(hx4)
        hx5 = self.rebnconv5(hx)
        hx6 = self.rebnconv6(hx5)
        hx5d = self.rebnconv5d(torch.cat((hx6, hx5), 1))
        hx5dup = _upsample_like(hx5d, hx4)
        hx4d = self.rebnconv4d(torch.cat((hx5dup, hx4), 1))
        hx4dup = _upsample_like(hx4d, hx3)
        hx3d = self.rebnconv3d(torch.cat((hx4dup, hx3), 1))
        hx3dup = _upsample_like(hx3d, hx2)
        hx2d = self.rebnconv2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)
        hx1d = self.rebnconv1d(torch.cat((hx2dup, hx1), 1))
        return hx1d + hxin

# RSU5 Module
class RSU5(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(RSU5, self).__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch, dirate=1)
        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool3 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.rebnconv5 = REBNCONV(mid_ch, mid_ch, dirate=2)
        self.rebnconv4d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv3d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv2d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv1d = REBNCONV(mid_ch*2, out_ch, dirate=1)

    def forward(self, x):
        hx = x
        hxin = self.rebnconvin(hx)
        hx1 = self.rebnconv1(hxin)
        hx = self.pool1(hx1)
        hx2 = self.rebnconv2(hx)
        hx = self.pool2(hx2)
        hx3 = self.rebnconv3(hx)
        hx = self.pool3(hx3)
        hx4 = self.rebnconv4(hx)
        hx5 = self.rebnconv5(hx4)
        hx4d = self.rebnconv4d(torch.cat((hx5, hx4), 1))
        hx4dup = _upsample_like(hx4d, hx3)
        hx3d = self.rebnconv3d(torch.cat((hx4dup, hx3), 1))
        hx3dup = _upsample_like(hx3d, hx2)
        hx2d = self.rebnconv2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)
        hx1d = self.rebnconv1d(torch.cat((hx2dup, hx1), 1))
        return hx1d + hxin

# RSU4 Module
class RSU4(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(RSU4, self).__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch, dirate=1)
        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)
        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dirate=2)
        self.rebnconv3d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv2d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv1d = REBNCONV(mid_ch*2, out_ch, dirate=1)

    def forward(self, x):
        hx = x
        hxin = self.rebnconvin(hx)
        hx1 = self.rebnconv1(hxin)
        hx = self.pool1(hx1)
        hx2 = self.rebnconv2(hx)
        hx = self.pool2(hx2)
        hx3 = self.rebnconv3(hx)
        hx4 = self.rebnconv4(hx3)
        hx3d = self.rebnconv3d(torch.cat((hx4, hx3), 1))
        hx3dup = _upsample_like(hx3d, hx2)
        hx2d = self.rebnconv2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)
        hx1d = self.rebnconv1d(torch.cat((hx2dup, hx1), 1))
        return hx1d + hxin

# RSU4F Module
class RSU4F(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(RSU4F, self).__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch, dirate=1)
        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dirate=1)
        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dirate=2)
        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dirate=4)
        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dirate=8)
        self.rebnconv3d = REBNCONV(mid_ch*2, mid_ch, dirate=4)
        self.rebnconv2d = REBNCONV(mid_ch*2, mid_ch, dirate=2)
        self.rebnconv1d = REBNCONV(mid_ch*2, out_ch, dirate=1)

    def forward(self, x):
        hx = x
        hxin = self.rebnconvin(hx)
        hx1 = self.rebnconv1(hxin)
        hx2 = self.rebnconv2(hx1)
        hx3 = self.rebnconv3(hx2)
        hx4 = self.rebnconv4(hx3)
        hx3d = self.rebnconv3d(torch.cat((hx4, hx3), 1))
        hx2d = self.rebnconv2d(torch.cat((hx3d, hx2), 1))
        hx1d = self.rebnconv1d(torch.cat((hx2d, hx1), 1))
        return hx1d + hxin

# Upsample helper function
def _upsample_like(src, target):
    return F.interpolate(src, size=target.shape[2:], mode='bilinear', align_corners=False)

# U²-Net Model Definition
class U2NET(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super(U2NET, self).__init__()
        self.stage1 = RSU7(in_ch, 32, 64)
        self.pool12 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.stage2 = RSU6(64, 32, 128)
        self.pool23 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.stage3 = RSU5(128, 64, 256)
        self.pool34 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.stage4 = RSU4(256, 128, 512)
        self.pool45 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.stage5 = RSU4F(512, 256, 512)
        self.pool56 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.stage6 = RSU4F(512, 256, 512)

        # decoder
        self.stage5d = RSU4F(1024, 256, 512)
        self.stage4d = RSU4(1024, 128, 256)
        self.stage3d = RSU5(512, 64, 128)
        self.stage2d = RSU6(256, 32, 64)
        self.stage1d = RSU7(128, 16, 64)  # This should output 64 channels

        self.side1 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.side2 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.side3 = nn.Conv2d(128, out_ch, 3, padding=1)
        self.side4 = nn.Conv2d(256, out_ch, 3, padding=1)
        self.side5 = nn.Conv2d(512, out_ch, 3, padding=1)
        self.side6 = nn.Conv2d(512, out_ch, 3, padding=1)

        self.outconv = nn.Conv2d(6*out_ch, out_ch, 1)

    def forward(self, x):
        hx = x

        # stage 1
        hx1 = self.stage1(hx)
        hx = self.pool12(hx1)

        # stage 2
        hx2 = self.stage2(hx)
        hx = self.pool23(hx2)

        # stage 3
        hx3 = self.stage3(hx)
        hx = self.pool34(hx3)

        # stage 4
        hx4 = self.stage4(hx)
        hx = self.pool45(hx4)

        # stage 5
        hx5 = self.stage5(hx)
        hx = self.pool56(hx5)

        # stage 6
        hx6 = self.stage6(hx)
        hx6up = _upsample_like(hx6, hx5)

        # decoder
        hx5d = self.stage5d(torch.cat((hx6up, hx5), 1))
        hx5dup = _upsample_like(hx5d, hx4)

        hx4d = self.stage4d(torch.cat((hx5dup, hx4), 1))
        hx4dup = _upsample_like(hx4d, hx3)

        hx3d = self.stage3d(torch.cat((hx4dup, hx3), 1))
        hx3dup = _upsample_like(hx3d, hx2)

        hx2d = self.stage2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)

        hx1d = self.stage1d(torch.cat((hx2dup, hx1), 1))

        # side output
        d1 = self.side1(hx1d)
        d2 = self.side2(hx2d)
        d2 = _upsample_like(d2, d1)
        d3 = self.side3(hx3d)
        d3 = _upsample_like(d3, d1)
        d4 = self.side4(hx4d)
        d4 = _upsample_like(d4, d1)
        d5 = self.side5(hx5d)
        d5 = _upsample_like(d5, d1)
        d6 = self.side6(hx6)
        d6 = _upsample_like(d6, d1)

        d0 = self.outconv(torch.cat((d1, d2, d3, d4, d5, d6), 1))

        return torch.sigmoid(d0), torch.sigmoid(d1), torch.sigmoid(d2), torch.sigmoid(d3), torch.sigmoid(d4), torch.sigmoid(d5), torch.sigmoid(d6)

# Download U²-Net model
def download_u2net_model(model_path):
    # URL for the pre-trained U²-Net model
    model_url = "https://drive.google.com/uc?id=1ao1ovG1Qtx4b7EoskHXmi2E9rp5CHLcZ"

    # Create directory if it doesn't exist
    os.makedirs(os.path.dirname(model_path), exist_ok=True)

    # Download the model
    print("Downloading U²-Net model...")
    gdown.download(model_url, model_path, quiet=False)
    print(f"Model downloaded to {model_path}")

# Load the pre-trained model
def load_u2net(model_path):
    # Download model if it doesn't exist
    if not os.path.exists(model_path):
        download_u2net_model(model_path)

    net = U2NET()
    if torch.cuda.is_available():
        net.load_state_dict(torch.load(model_path), strict=False)  # Use strict=False to handle mismatches
        net.cuda()
    else:
        net.load_state_dict(torch.load(model_path, map_location='cpu'), strict=False)
    net.eval()
    return net

# Preprocessing function
def preprocess_image(image_path, size=320):
    image = Image.open(image_path).convert('RGB')
    w, h = image.size

    # Resize while maintaining aspect ratio
    scale = size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    image = image.resize((new_w, new_h), Image.BICUBIC)

    # Pad to make it square
    padded_image = Image.new('RGB', (size, size), (0, 0, 0))
    padded_image.paste(image, ((size - new_w) // 2, (size - new_h) // 2))

    # Convert to tensor and normalize
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    input_tensor = transform(padded_image).unsqueeze(0)
    return input_tensor, (w, h), (new_w, new_h), ((size - new_w) // 2, (size - new_h) // 2)

# Post-processing function
def postprocess_mask(mask_tensor, original_size, new_size, padding):
    mask = mask_tensor.squeeze().cpu().numpy()

    # Remove padding
    pad_x, pad_y = padding
    new_w, new_h = new_size
    mask = mask[pad_y:pad_y+new_h, pad_x:pad_x+new_w]

    # Resize to original dimensions
    orig_w, orig_h = original_size
    mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

    # Apply threshold to create binary mask
    _, binary_mask = cv2.threshold(mask, 0.5, 1, cv2.THRESH_BINARY)

    # Clean up the mask
    binary_mask = morphology.remove_small_objects(binary_mask.astype(bool), min_size=50)
    binary_mask = morphology.remove_small_holes(binary_mask, area_threshold=50)

    return binary_mask.astype(np.uint8) * 255

# Generate mask for a single image
def generate_mask(model, image_path, output_path=None):
    # Preprocess image
    input_tensor, original_size, new_size, padding = preprocess_image(image_path)

    if torch.cuda.is_available():
        input_tensor = input_tensor.cuda()

    # Run inference
    with torch.no_grad():
        d0, d1, d2, d3, d4, d5, d6 = model(input_tensor)
        pred = d0[:, 0, :, :]

    # Post-process mask
    mask = postprocess_mask(pred, original_size, new_size, padding)

    # Save mask if output path is provided
    if output_path:
        cv2.imwrite(output_path, mask)

    return mask

# Process a directory of images
def process_directory(model, input_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
    image_files = [f for f in os.listdir(input_dir)
                  if os.path.splitext(f)[1].lower() in image_extensions]

    for image_file in image_files:
        image_path = os.path.join(input_dir, image_file)
        output_path = os.path.join(output_dir, f"mask_{os.path.splitext(image_file)[0]}.png")

        print(f"Processing {image_file}...")
        try:
            generate_mask(model, image_path, output_path)
            print(f"Saved mask to {output_path}")
        except Exception as e:
            print(f"Error processing {image_file}: {str(e)}")

# Manual refinement tools (optional)
def refine_mask(mask_path, output_path):
    """Basic refinement function - can be expanded based on specific needs"""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    # Apply morphological operations to clean up the mask
    kernel = np.ones((3, 3), np.uint8)
    refined_mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    refined_mask = cv2.morphologyEx(refined_mask, cv2.MORPH_OPEN, kernel)

    cv2.imwrite(output_path, refined_mask)
    return refined_mask

# Main function
def main():
    # Path to your trained U²-Net model
    model_path = "/content/u2net_model.pth"

    # Input and output directories
    input_dir = "/content/Original" # Corrected input directory
    output_dir = "/content/output/masks"
    refined_dir = "/content/refined/masks"  # Optional

    # Load model
    print("Loading model...")
    model = load_u2net(model_path)

    # Generate masks
    print("Generating masks...")
    process_directory(model, input_dir, output_dir)

    # Optional: Refine masks
    print("Refining masks...")
    if not os.path.exists(refined_dir):
        os.makedirs(refined_dir)

    mask_files = [f for f in os.listdir(output_dir) if f.endswith('.png')]
    for mask_file in mask_files:
        mask_path = os.path.join(output_dir, mask_file)
        refined_path = os.path.join(refined_dir, f"refined_{mask_file}")
        refine_mask(mask_path, refined_path)

    print("Process completed!")

if __name__ == "__main__":
    main()

Loading model...


Downloading...
From (original): https://drive.google.com/uc?id=1ao1ovG1Qtx4b7EoskHXmi2E9rp5CHLcZ
From (redirected): https://drive.google.com/uc?id=1ao1ovG1Qtx4b7EoskHXmi2E9rp5CHLcZ&confirm=t&uuid=340f7a07-551f-43c2-aa75-1f27966f617b
To: /content/u2net_model.pth
100%|██████████| 176M/176M [00:01<00:00, 132MB/s]


Model downloaded to /content/u2net_model.pth
Generating masks...
Processing Whole_062_jpg.rf.cb031d362fdf3a2a6c376c019010cc97.jpg...
Saved mask to /content/output/masks/mask_Whole_062_jpg.rf.cb031d362fdf3a2a6c376c019010cc97.png
Processing Mix_053_jpg.rf.71678349351eb7a1bece1f98c73dfec6.jpg...
Saved mask to /content/output/masks/mask_Mix_053_jpg.rf.71678349351eb7a1bece1f98c73dfec6.png
Processing Whole_021_jpg.rf.7d6cd60fd1ef7b833c3e9c17475c55fd.jpg...
Saved mask to /content/output/masks/mask_Whole_021_jpg.rf.7d6cd60fd1ef7b833c3e9c17475c55fd.png
Processing Whole_006_jpg.rf.84a4744e58e501665229e28363931ed9.jpg...
Saved mask to /content/output/masks/mask_Whole_006_jpg.rf.84a4744e58e501665229e28363931ed9.png
Processing Beads_058_jpg.rf.5ab48798a07323246667565082460afa.jpg...
Saved mask to /content/output/masks/mask_Beads_058_jpg.rf.5ab48798a07323246667565082460afa.png
Processing Whole_075_jpg.rf.b5b1d3893d210f09ce9c671126732661.jpg...
Saved mask to /content/output/masks/mask_Whole_075_jpg.r

KeyboardInterrupt: 

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from skimage import morphology, measure, filters
from skimage.feature import peak_local_max  # Add this import
from skimage.segmentation import watershed  # Add this import if not already imported
import urllib.request
import gdown
from scipy import ndimage

# Enhanced preprocessing for microplastics
def preprocess_image_microplastics(image_path, size=512):  # Increased size for better small object detection
    image = Image.open(image_path).convert('RGB')
    w, h = image.size

    # Convert to numpy for enhancement
    img_array = np.array(image)

    # Apply contrast enhancement for microplastics
    lab = cv2.cvtColor(img_array, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    # CLAHE (Contrast Limited Adaptive Histogram Equalization)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    enhanced_img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    image = Image.fromarray(enhanced_img)

    # Resize while maintaining aspect ratio (larger size for small objects)
    scale = size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    image = image.resize((new_w, new_h), Image.LANCZOS)  # Better interpolation

    # Pad to make it square
    padded_image = Image.new('RGB', (size, size), (0, 0, 0))
    padded_image.paste(image, ((size - new_w) // 2, (size - new_h) // 2))

    # Convert to tensor and normalize
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    input_tensor = transform(padded_image).unsqueeze(0)
    return input_tensor, (w, h), (new_w, new_h), ((size - new_w) // 2, (size - new_h) // 2)

# Enhanced post-processing for microplastics
def postprocess_mask_microplastics(mask_tensor, original_size, new_size, padding, original_image_path=None):
    mask = mask_tensor.squeeze().cpu().numpy()

    # Remove padding
    pad_x, pad_y = padding
    new_w, new_h = new_size
    mask = mask[pad_y:pad_y+new_h, pad_x:pad_x+new_w]

    # Resize to original dimensions
    orig_w, orig_h = original_size
    mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)

    # Adaptive thresholding for microplastics
    binary_mask = np.zeros_like(mask, dtype=np.uint8)

    # Use adaptive thresholding to better capture small objects
    if np.max(mask) > 0:  # Only if there are non-zero values
        # Adaptive threshold
        adaptive_thresh = cv2.adaptiveThreshold(
            (mask * 255).astype(np.uint8),
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            11,  # Smaller block size for small objects
            2     # Lower constant
        )
        binary_mask = (adaptive_thresh > 0).astype(np.uint8)

    # Additional processing if original image is available for context
    if original_image_path:
        original_img = cv2.imread(original_image_path, cv2.IMREAD_GRAYSCALE)
        if original_img is not None:
            # Use edge information to refine mask
            edges = cv2.Canny(original_img, 50, 150)
            binary_mask = np.logical_or(binary_mask, edges > 0).astype(np.uint8)

    # Enhanced morphological operations for microplastics
    kernel = np.ones((2, 2), np.uint8)  # Smaller kernel for microplastics

    # Remove noise while preserving small objects
    binary_mask = morphology.remove_small_objects(binary_mask.astype(bool), min_size=10)  # Smaller min size
    binary_mask = morphology.remove_small_holes(binary_mask, area_threshold=20)  # Smaller hole threshold

    # Gentle closing to connect nearby microplastics without merging them
    binary_mask = cv2.morphologyEx(binary_mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)

    # Final cleanup
    binary_mask = morphology.remove_small_objects(binary_mask.astype(bool), min_size=5)

    return binary_mask.astype(np.uint8) * 255

# Multi-scale processing for better microplastic detection
def generate_mask_multi_scale(model, image_path, output_path=None, scales=[0.8, 1.0, 1.2]):
    """Process image at multiple scales and combine results"""
    all_masks = []

    for scale_factor in scales:
        # Load and preprocess at different scale
        image = Image.open(image_path).convert('RGB')
        w, h = image.size

        # Apply scale
        new_size = (int(w * scale_factor), int(h * scale_factor))
        scaled_image = image.resize(new_size, Image.LANCZOS)

        # Convert to tensor
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        input_tensor = transform(scaled_image).unsqueeze(0)

        if torch.cuda.is_available():
            input_tensor = input_tensor.cuda()

        # Run inference
        with torch.no_grad():
            d0, d1, d2, d3, d4, d5, d6 = model(input_tensor)
            pred = d0[:, 0, :, :]

        # Post-process and resize to original size
        mask = pred.squeeze().cpu().numpy()
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_CUBIC)
        all_masks.append(mask)

    # Combine masks from different scales
    combined_mask = np.mean(all_masks, axis=0)

    # Final thresholding
    _, binary_mask = cv2.threshold(combined_mask, 0.3, 1, cv2.THRESH_BINARY)  # Lower threshold

    # Enhanced cleanup for microplastics
    binary_mask = morphology.remove_small_objects(binary_mask.astype(bool), min_size=8)
    binary_mask = morphology.remove_small_holes(binary_mask, area_threshold=15)

    binary_mask = binary_mask.astype(np.uint8) * 255

    # Save if output path provided
    if output_path:
        cv2.imwrite(output_path, binary_mask)

    return binary_mask

# Enhanced refinement specifically for microplastics
def refine_microplastic_mask(mask_path, original_image_path, output_path):
    """Advanced refinement for microplastic masks"""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    original = cv2.imread(original_image_path, cv2.IMREAD_GRAYSCALE)

    if original is None:
        original = cv2.imread(original_image_path, cv2.IMREAD_COLOR)
        original = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)

    # Enhance contrast of original image
    original = cv2.equalizeHist(original)

    # Use edge detection to find microplastic boundaries
    edges = cv2.Canny(original, 30, 100)

    # Combine with original mask
    refined_mask = np.logical_or(mask > 0, edges > 0).astype(np.uint8) * 255

    # Watershed segmentation for better separation
    distance = ndimage.distance_transform_edt(refined_mask > 0)

    # Corrected peak_local_max usage
    local_maxi = peak_local_max(
        distance,
        exclude_border=False,  # Changed from indices=False
        footprint=np.ones((3, 3)),
        labels=refined_mask > 0
    )

    # Convert coordinates to boolean mask
    local_maxi_mask = np.zeros_like(distance, dtype=bool)
    local_maxi_mask[tuple(local_maxi.T)] = True

    markers = ndimage.label(local_maxi_mask)[0]
    labels = watershed(-distance, markers, mask=refined_mask > 0)

    # Convert back to binary
    refined_mask = (labels > 0).astype(np.uint8) * 255

    # Final cleanup
    kernel = np.ones((2, 2), np.uint8)
    refined_mask = cv2.morphologyEx(refined_mask, cv2.MORPH_CLOSE, kernel)
    refined_mask = cv2.morphologyEx(refined_mask, cv2.MORPH_OPEN, kernel)

    cv2.imwrite(output_path, refined_mask)
    return refined_mask

# Modified process_directory for microplastics
def process_directory_microplastics(model, input_dir, output_dir, use_multi_scale=True):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
    image_files = [f for f in os.listdir(input_dir)
                  if os.path.splitext(f)[1].lower() in image_extensions]

    for image_file in image_files:
        image_path = os.path.join(input_dir, image_file)
        output_path = os.path.join(output_dir, f"mask_{os.path.splitext(image_file)[0]}.png")

        print(f"Processing {image_file} for microplastics...")
        try:
            if use_multi_scale:
                mask = generate_mask_multi_scale(model, image_path, output_path)
            else:
                # Enhanced single-scale processing
                input_tensor, original_size, new_size, padding = preprocess_image_microplastics(image_path)

                if torch.cuda.is_available():
                    input_tensor = input_tensor.cuda()

                with torch.no_grad():
                    d0, d1, d2, d3, d4, d5, d6 = model(input_tensor)
                    pred = d0[:, 0, :, :]

                mask = postprocess_mask_microplastics(pred, original_size, new_size, padding, image_path)
                cv2.imwrite(output_path, mask)

            print(f"Saved microplastic mask to {output_path}")
        except Exception as e:
            print(f"Error processing {image_file}: {str(e)}")

# Main function optimized for microplastics
def main():
    # Path to your trained U²-Net model
    model_path = "/content/u2net_model.pth"

    # Input and output directories
    input_dir = "/content/Original"
    output_dir = "/content/output1/microplastic_masks"
    refined_dir = "/content/refined1/microplastic_masks"

    # Load model
    print("Loading model for microplastic detection...")
    model = load_u2net(model_path)

    # Generate masks with microplastic-optimized processing
    print("Generating microplastic masks...")
    process_directory_microplastics(model, input_dir, output_dir, use_multi_scale=True)

    # Enhanced refinement for microplastics
    print("Refining microplastic masks...")
    if not os.path.exists(refined_dir):
        os.makedirs(refined_dir)

    mask_files = [f for f in os.listdir(output_dir) if f.endswith('.png')]
    image_files = [f for f in os.listdir(input_dir)
                  if os.path.splitext(f)[1].lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']]

    for mask_file in mask_files:
        base_name = mask_file.replace('mask_', '').replace('.png', '')
        matching_images = [f for f in image_files if base_name in f]

        if matching_images:
            original_image = matching_images[0]
            mask_path = os.path.join(output_dir, mask_file)
            original_path = os.path.join(input_dir, original_image)
            refined_path = os.path.join(refined_dir, f"refined_{mask_file}")

            try:
                refine_microplastic_mask(mask_path, original_path, refined_path)
                print(f"Refined {mask_file}")
            except Exception as e:
                print(f"Error refining {mask_file}: {str(e)}")

    print("Microplastic detection process completed!")

if __name__ == "__main__":
    main()

Loading model for microplastic detection...
Generating microplastic masks...
Processing Whole_062_jpg.rf.cb031d362fdf3a2a6c376c019010cc97.jpg for microplastics...
Saved microplastic mask to /content/output1/microplastic_masks/mask_Whole_062_jpg.rf.cb031d362fdf3a2a6c376c019010cc97.png
Processing Mix_053_jpg.rf.71678349351eb7a1bece1f98c73dfec6.jpg for microplastics...
Saved microplastic mask to /content/output1/microplastic_masks/mask_Mix_053_jpg.rf.71678349351eb7a1bece1f98c73dfec6.png
Processing Whole_021_jpg.rf.7d6cd60fd1ef7b833c3e9c17475c55fd.jpg for microplastics...
Saved microplastic mask to /content/output1/microplastic_masks/mask_Whole_021_jpg.rf.7d6cd60fd1ef7b833c3e9c17475c55fd.png
Processing Whole_006_jpg.rf.84a4744e58e501665229e28363931ed9.jpg for microplastics...
Saved microplastic mask to /content/output1/microplastic_masks/mask_Whole_006_jpg.rf.84a4744e58e501665229e28363931ed9.png
Processing Beads_058_jpg.rf.5ab48798a07323246667565082460afa.jpg for microplastics...
Saved mic

In [ ]:
!unzip /content/test.zip

Archive:  /content/test.zip
  inflating: test/_annotations.coco.json  
  inflating: test/Beads_028_jpg.rf.f420c2a5033bba22e0fe06faf6e31644.jpg  
  inflating: test/Beads_054_jpg.rf.652d634375e1f167d67e1cf07b7c72ed.jpg  
  inflating: test/Broken_157_jpg.rf.ae621dcfb80cfa96b59a94ba9830ce32.jpg  
  inflating: test/Broken_162_jpg.rf.e867444a3993086374f4487aec5d8500.jpg  
  inflating: test/Broken_169_jpg.rf.7751f4154264d4c4bce12a8ebb3a6c7f.jpg  
  inflating: test/Broken_179_jpg.rf.92497031eb0c82965434579f2e1477c7.jpg  
  inflating: test/Broken_182_jpg.rf.ac488b5d9d27234f7b68018cbb51fd82.jpg  
  inflating: test/Mix_050_jpg.rf.bff52bea2f67e983781f241f6662b663.jpg  
  inflating: test/Mix_063_jpg.rf.a02483f5b4498eeeeb119f0ee5f9cb97.jpg  
  inflating: test/Mix_065_jpg.rf.ae43c6d55e48090ae13e2388b2d8a515.jpg  
  inflating: test/Mix_076_jpg.rf.5a3add17264accc0090f07e2e6480265.jpg  
  inflating: test/Mix_081_jpg.rf.82bd2bcc4f1b7b3037364389bd221bd7.jpg  
  inflating: test/Mix_082_jpg.rf.048f2c3a730fc

In [ ]:
!unzip /content/valid.zip

Archive:  /content/valid.zip
  inflating: valid/_annotations.coco.json  
  inflating: valid/Beads_001_jpg.rf.8724c474d820ee0ad7c72c0755199a3e.jpg  
  inflating: valid/Beads_006_jpg.rf.026760540a93088ede7905e81333c8be.jpg  
  inflating: valid/Beads_009_jpg.rf.a09846f22e3d83f26027ced183a852a7.jpg  
  inflating: valid/Beads_014_jpg.rf.e435c4c050d4d9026885a4c7de1fe130.jpg  
  inflating: valid/Beads_015_jpg.rf.2d048f175128bc9b6fc9542b10a9dea9.jpg  
  inflating: valid/Beads_020_jpg.rf.7d3d0471f876c6afcd1f3dbfdf95f086.jpg  
  inflating: valid/Beads_022_jpg.rf.e068c726e6c831630a6f2563aa9a5eff.jpg  
  inflating: valid/Beads_035_jpg.rf.c7a862d8de466ff3b4b024cce28cc612.jpg  
  inflating: valid/Beads_039_jpg.rf.3faf365a370bdd10cad1b7ada5985b9b.jpg  
  inflating: valid/Beads_042_jpg.rf.cf16387ff0bebf789a3140af2f17d6d7.jpg  
  inflating: valid/Beads_044_jpg.rf.57209175b35323aa000f604672292fe0.jpg  
  inflating: valid/Beads_051_jpg.rf.a59cfdbe8df9440fbc74498c3b473dfc.jpg  
  inflating: valid/Beads_05

In [ ]:
len(os.listdir('/content/train'))

827

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from skimage import morphology, measure, filters
import urllib.request
import gdown
from scipy import ndimage
from skimage.feature import peak_local_max
from skimage.segmentation import watershed

# U^2-Net Model Architecture (must match your trained model)
class U2NET(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super(U2NET, self).__init__()

        # Encoder
        self.stage1 = RSU7(in_ch, 32, 64)
        self.pool12 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.stage2 = RSU6(64, 32, 128)
        self.pool23 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.stage3 = RSU5(128, 64, 256)
        self.pool34 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.stage4 = RSU4(256, 128, 512)
        self.pool45 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.stage5 = RSU4F(512, 256, 512)
        self.pool56 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.stage6 = RSU4F(512, 256, 512)

        # Decoder
        self.stage5d = RSU4F(1024, 256, 512)
        self.stage4d = RSU4(1024, 128, 256)
        self.stage3d = RSU5(512, 64, 128)
        self.stage2d = RSU6(256, 32, 64)
        self.stage1d = RSU7(128, 16, 64)

        self.side1 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.side2 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.side3 = nn.Conv2d(128, out_ch, 3, padding=1)
        self.side4 = nn.Conv2d(256, out_ch, 3, padding=1)
        self.side5 = nn.Conv2d(512, out_ch, 3, padding=1)
        self.side6 = nn.Conv2d(512, out_ch, 3, padding=1)

        self.outconv = nn.Conv2d(6*out_ch, out_ch, 1)

    def forward(self, x):
        hx = x

        # Encoder
        hx1 = self.stage1(hx)
        hx = self.pool12(hx1)

        hx2 = self.stage2(hx)
        hx = self.pool23(hx2)

        hx3 = self.stage3(hx)
        hx = self.pool34(hx3)

        hx4 = self.stage4(hx)
        hx = self.pool45(hx4)

        hx5 = self.stage5(hx)
        hx = self.pool56(hx5)

        hx6 = self.stage6(hx)
        hx6up = _upsample_like(hx6, hx5)

        # Decoder
        hx5d = self.stage5d(torch.cat((hx6up, hx5), 1))
        hx5dup = _upsample_like(hx5d, hx4)

        hx4d = self.stage4d(torch.cat((hx5dup, hx4), 1))
        hx4dup = _upsample_like(hx4d, hx3)

        hx3d = self.stage3d(torch.cat((hx4dup, hx3), 1))
        hx3dup = _upsample_like(hx3d, hx2)

        hx2d = self.stage2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)

        hx1d = self.stage1d(torch.cat((hx2dup, hx1), 1))

        # Side outputs
        d1 = self.side1(hx1d)
        d2 = self.side2(hx2d)
        d2 = _upsample_like(d2, d1)
        d3 = self.side3(hx3d)
        d3 = _upsample_like(d3, d1)
        d4 = self.side4(hx4d)
        d4 = _upsample_like(d4, d1)
        d5 = self.side5(hx5d)
        d5 = _upsample_like(d5, d1)
        d6 = self.side6(hx6)
        d6 = _upsample_like(d6, d1)

        d0 = self.outconv(torch.cat((d1, d2, d3, d4, d5, d6), 1))

        return d0, d1, d2, d3, d4, d5, d6

# Helper modules for U^2-Net
class RSU7(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(RSU7, self).__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch, dirate=1)

        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool3 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool4 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv5 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool5 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv6 = REBNCONV(mid_ch, mid_ch, dirate=1)

        self.rebnconv7 = REBNCONV(mid_ch, mid_ch, dirate=2)

        self.rebnconv6d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv5d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv4d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv3d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv2d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv1d = REBNCONV(mid_ch*2, out_ch, dirate=1)

    def forward(self, x):
        hx = x
        hxin = self.rebnconvin(hx)

        hx1 = self.rebnconv1(hxin)
        hx = self.pool1(hx1)

        hx2 = self.rebnconv2(hx)
        hx = self.pool2(hx2)

        hx3 = self.rebnconv3(hx)
        hx = self.pool3(hx3)

        hx4 = self.rebnconv4(hx)
        hx = self.pool4(hx4)

        hx5 = self.rebnconv5(hx)
        hx = self.pool5(hx5)

        hx6 = self.rebnconv6(hx)

        hx7 = self.rebnconv7(hx6)

        hx6d = self.rebnconv6d(torch.cat((hx7, hx6), 1))
        hx6dup = _upsample_like(hx6d, hx5)

        hx5d = self.rebnconv5d(torch.cat((hx6dup, hx5), 1))
        hx5dup = _upsample_like(hx5d, hx4)

        hx4d = self.rebnconv4d(torch.cat((hx5dup, hx4), 1))
        hx4dup = _upsample_like(hx4d, hx3)

        hx3d = self.rebnconv3d(torch.cat((hx4dup, hx3), 1))
        hx3dup = _upsample_like(hx3d, hx2)

        hx2d = self.rebnconv2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)

        hx1d = self.rebnconv1d(torch.cat((hx2dup, hx1), 1))

        return hx1d + hxin

class RSU6(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(RSU6, self).__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch, dirate=1)

        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool3 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool4 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv5 = REBNCONV(mid_ch, mid_ch, dirate=1)

        self.rebnconv6 = REBNCONV(mid_ch, mid_ch, dirate=2)

        self.rebnconv5d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv4d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv3d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv2d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv1d = REBNCONV(mid_ch*2, out_ch, dirate=1)

    def forward(self, x):
        hx = x
        hxin = self.rebnconvin(hx)

        hx1 = self.rebnconv1(hxin)
        hx = self.pool1(hx1)

        hx2 = self.rebnconv2(hx)
        hx = self.pool2(hx2)

        hx3 = self.rebnconv3(hx)
        hx = self.pool3(hx3)

        hx4 = self.rebnconv4(hx)
        hx = self.pool4(hx4)

        hx5 = self.rebnconv5(hx)

        hx6 = self.rebnconv6(hx5)

        hx5d = self.rebnconv5d(torch.cat((hx6, hx5), 1))
        hx5dup = _upsample_like(hx5d, hx4)

        hx4d = self.rebnconv4d(torch.cat((hx5dup, hx4), 1))
        hx4dup = _upsample_like(hx4d, hx3)

        hx3d = self.rebnconv3d(torch.cat((hx4dup, hx3), 1))
        hx3dup = _upsample_like(hx3d, hx2)

        hx2d = self.rebnconv2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)

        hx1d = self.rebnconv1d(torch.cat((hx2dup, hx1), 1))

        return hx1d + hxin

class RSU5(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(RSU5, self).__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch, dirate=1)

        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool3 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dirate=1)

        self.rebnconv5 = REBNCONV(mid_ch, mid_ch, dirate=2)

        self.rebnconv4d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv3d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv2d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv1d = REBNCONV(mid_ch*2, out_ch, dirate=1)

    def forward(self, x):
        hx = x
        hxin = self.rebnconvin(hx)

        hx1 = self.rebnconv1(hxin)
        hx = self.pool1(hx1)

        hx2 = self.rebnconv2(hx)
        hx = self.pool2(hx2)

        hx3 = self.rebnconv3(hx)
        hx = self.pool3(hx3)

        hx4 = self.rebnconv4(hx)

        hx5 = self.rebnconv5(hx4)

        hx4d = self.rebnconv4d(torch.cat((hx5, hx4), 1))
        hx4dup = _upsample_like(hx4d, hx3)

        hx3d = self.rebnconv3d(torch.cat((hx4dup, hx3), 1))
        hx3dup = _upsample_like(hx3d, hx2)

        hx2d = self.rebnconv2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)

        hx1d = self.rebnconv1d(torch.cat((hx2dup, hx1), 1))

        return hx1d + hxin

class RSU4(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(RSU4, self).__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch, dirate=1)

        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dirate=1)

        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dirate=2)

        self.rebnconv3d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv2d = REBNCONV(mid_ch*2, mid_ch, dirate=1)
        self.rebnconv1d = REBNCONV(mid_ch*2, out_ch, dirate=1)

    def forward(self, x):
        hx = x
        hxin = self.rebnconvin(hx)

        hx1 = self.rebnconv1(hxin)
        hx = self.pool1(hx1)

        hx2 = self.rebnconv2(hx)
        hx = self.pool2(hx2)

        hx3 = self.rebnconv3(hx)

        hx4 = self.rebnconv4(hx3)

        hx3d = self.rebnconv3d(torch.cat((hx4, hx3), 1))
        hx3dup = _upsample_like(hx3d, hx2)

        hx2d = self.rebnconv2d(torch.cat((hx3dup, hx2), 1))
        hx2dup = _upsample_like(hx2d, hx1)

        hx1d = self.rebnconv1d(torch.cat((hx2dup, hx1), 1))

        return hx1d + hxin

class RSU4F(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(RSU4F, self).__init__()
        self.rebnconvin = REBNCONV(in_ch, out_ch, dirate=1)

        self.rebnconv1 = REBNCONV(out_ch, mid_ch, dirate=1)
        self.rebnconv2 = REBNCONV(mid_ch, mid_ch, dirate=2)
        self.rebnconv3 = REBNCONV(mid_ch, mid_ch, dirate=4)

        self.rebnconv4 = REBNCONV(mid_ch, mid_ch, dirate=8)

        self.rebnconv3d = REBNCONV(mid_ch*2, mid_ch, dirate=4)
        self.rebnconv2d = REBNCONV(mid_ch*2, mid_ch, dirate=2)
        self.rebnconv1d = REBNCONV(mid_ch*2, out_ch, dirate=1)

    def forward(self, x):
        hx = x
        hxin = self.rebnconvin(hx)

        hx1 = self.rebnconv1(hxin)
        hx2 = self.rebnconv2(hx1)
        hx3 = self.rebnconv3(hx2)

        hx4 = self.rebnconv4(hx3)

        hx3d = self.rebnconv3d(torch.cat((hx4, hx3), 1))
        hx2d = self.rebnconv2d(torch.cat((hx3d, hx2), 1))
        hx1d = self.rebnconv1d(torch.cat((hx2d, hx1), 1))

        return hx1d + hxin

class REBNCONV(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, dirate=1):
        super(REBNCONV, self).__init__()
        self.conv_s1 = nn.Conv2d(in_ch, out_ch, 3, padding=1*dirate, dilation=1*dirate)
        self.bn_s1 = nn.BatchNorm2d(out_ch)
        self.relu_s1 = nn.ReLU(inplace=True)

    def forward(self, x):
        hx = x
        xout = self.relu_s1(self.bn_s1(self.conv_s1(hx)))
        return xout

def _upsample_like(src, tar):
    src = F.interpolate(src, size=tar.shape[2:], mode='bilinear', align_corners=False)
    return src

# Enhanced preprocessing for microplastics
def preprocess_image_microplastics(image_path, size=512):  # Increased size for better small object detection
    image = Image.open(image_path).convert('RGB')
    w, h = image.size

    # Convert to numpy for enhancement
    img_array = np.array(image)

    # Apply contrast enhancement for microplastics
    lab = cv2.cvtColor(img_array, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    # CLAHE (Contrast Limited Adaptive Histogram Equalization)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    enhanced_img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    image = Image.fromarray(enhanced_img)

    # Resize while maintaining aspect ratio (larger size for small objects)
    scale = size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    image = image.resize((new_w, new_h), Image.LANCZOS)  # Better interpolation

    # Pad to make it square
    padded_image = Image.new('RGB', (size, size), (0, 0, 0))
    padded_image.paste(image, ((size - new_w) // 2, (size - new_h) // 2))

    # Convert to tensor and normalize
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    input_tensor = transform(padded_image).unsqueeze(0)
    return input_tensor, (w, h), (new_w, new_h), ((size - new_w) // 2, (size - new_h) // 2)

# Enhanced post-processing for microplastics
def postprocess_mask_microplastics(mask_tensor, original_size, new_size, padding, original_image_path=None):
    mask = mask_tensor.squeeze().cpu().numpy()

    # Remove padding
    pad_x, pad_y = padding
    new_w, new_h = new_size
    mask = mask[pad_y:pad_y+new_h, pad_x:pad_x+new_w]

    # Resize to original dimensions
    orig_w, orig_h = original_size
    mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)

    # Adaptive thresholding for microplastics
    binary_mask = np.zeros_like(mask, dtype=np.uint8)

    # Use adaptive thresholding to better capture small objects
    if np.max(mask) > 0:  # Only if there are non-zero values
        # Adaptive threshold
        adaptive_thresh = cv2.adaptiveThreshold(
            (mask * 255).astype(np.uint8),
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            11,  # Smaller block size for small objects
            2     # Lower constant
        )
        binary_mask = (adaptive_thresh > 0).astype(np.uint8)

    # Additional processing if original image is available for context
    if original_image_path:
        original_img = cv2.imread(original_image_path, cv2.IMREAD_GRAYSCALE)
        if original_img is not None:
            # Use edge information to refine mask
            edges = cv2.Canny(original_img, 50, 150)
            binary_mask = np.logical_or(binary_mask, edges > 0).astype(np.uint8)

    # Enhanced morphological operations for microplastics
    kernel = np.ones((2, 2), np.uint8)  # Smaller kernel for microplastics

    # Remove noise while preserving small objects
    binary_mask = morphology.remove_small_objects(binary_mask.astype(bool), min_size=10)  # Smaller min size
    binary_mask = morphology.remove_small_holes(binary_mask, area_threshold=20)  # Smaller hole threshold

    # Gentle closing to connect nearby microplastics without merging them
    binary_mask = cv2.morphologyEx(binary_mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel)

    # Final cleanup
    binary_mask = morphology.remove_small_objects(binary_mask.astype(bool), min_size=5)

    return binary_mask.astype(np.uint8) * 255

# Multi-scale processing for better microplastic detection
def generate_mask_multi_scale(model, image_path, output_path=None, scales=[0.8, 1.0, 1.2]):
    """Process image at multiple scales and combine results"""
    all_masks = []

    for scale_factor in scales:
        # Load and preprocess at different scale
        image = Image.open(image_path).convert('RGB')
        w, h = image.size

        # Apply scale
        new_size = (int(w * scale_factor), int(h * scale_factor))
        scaled_image = image.resize(new_size, Image.LANCZOS)

        # Convert to tensor
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        input_tensor = transform(scaled_image).unsqueeze(0)

        if torch.cuda.is_available():
            input_tensor = input_tensor.cuda()

        # Run inference
        with torch.no_grad():
            d0, d1, d2, d3, d4, d5, d6 = model(input_tensor)
            pred = d0[:, 0, :, :]

        # Post-process and resize to original size
        mask = pred.squeeze().cpu().numpy()
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_CUBIC)
        all_masks.append(mask)

    # Combine masks from different scales
    combined_mask = np.mean(all_masks, axis=0)

    # Final thresholding
    _, binary_mask = cv2.threshold(combined_mask, 0.3, 1, cv2.THRESH_BINARY)  # Lower threshold

    # Enhanced cleanup for microplastics
    binary_mask = morphology.remove_small_objects(binary_mask.astype(bool), min_size=8)
    binary_mask = morphology.remove_small_holes(binary_mask, area_threshold=15)

    binary_mask = binary_mask.astype(np.uint8) * 255

    # Save if output path provided
    if output_path:
        cv2.imwrite(output_path, binary_mask)

    return binary_mask

# Enhanced refinement specifically for microplastics
def refine_microplastic_mask(mask_path, original_image_path, output_path):
    """Advanced refinement for microplastic masks"""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    original = cv2.imread(original_image_path, cv2.IMREAD_GRAYSCALE)

    if original is None:
        original = cv2.imread(original_image_path, cv2.IMREAD_COLOR)
        original = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)

    # Enhance contrast of original image
    original = cv2.equalizeHist(original)

    # Use edge detection to find microplastic boundaries
    edges = cv2.Canny(original, 30, 100)

    # Combine with original mask
    refined_mask = np.logical_or(mask > 0, edges > 0).astype(np.uint8) * 255

    # Watershed segmentation for better separation
    distance = ndimage.distance_transform_edt(refined_mask > 0)
    local_maxi = peak_local_max(distance, indices=False, footprint=np.ones((3, 3)), labels=refined_mask > 0)
    markers = ndimage.label(local_maxi)[0]
    labels = watershed(-distance, markers, mask=refined_mask > 0)

    # Convert back to binary
    refined_mask = (labels > 0).astype(np.uint8) * 255

    # Final cleanup
    kernel = np.ones((2, 2), np.uint8)
    refined_mask = cv2.morphologyEx(refined_mask, cv2.MORPH_CLOSE, kernel)
    refined_mask = cv2.morphologyEx(refined_mask, cv2.MORPH_OPEN, kernel)

    cv2.imwrite(output_path, refined_mask)
    return refined_mask

# Load U2-Net model
def load_u2net(model_path):
    """Load the U2-Net model with proper error handling"""
    try:
        # Initialize model
        model = U2NET()

        # Load model weights
        if torch.cuda.is_available():
            model.load_state_dict(torch.load(model_path))
            model.cuda()
        else:
            model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

        model.eval()
        print(f"Model loaded successfully from {model_path}")
        return model
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        # If the model file doesn't exist, try to download it
        print("Attempting to download U2-Net model...")
        try:
            # URL to a pre-trained U2-Net model
            model_url = "https://github.com/NathanUA/U-2-Net/raw/master/saved_models/u2net.pth"
            urllib.request.urlretrieve(model_url, model_path)
            print("Model downloaded successfully.")

            # Try loading again
            if torch.cuda.is_available():
                model.load_state_dict(torch.load(model_path))
                model.cuda()
            else:
                model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

            model.eval()
            return model
        except Exception as e2:
            print(f"Failed to download model: {str(e2)}")
            return None

# Modified process_directory for microplastics
def process_directory_microplastics(model, input_dir, output_dir, use_multi_scale=True):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
    image_files = [f for f in os.listdir(input_dir)
                  if os.path.splitext(f)[1].lower() in image_extensions]

    for image_file in image_files:
        image_path = os.path.join(input_dir, image_file)
        # Use the original filename without "mask_" prefix
        output_path = os.path.join(output_dir, f"{os.path.splitext(image_file)[0]}.png")

        print(f"Processing {image_file} for microplastics...")
        try:
            if use_multi_scale:
                mask = generate_mask_multi_scale(model, image_path, output_path)
            else:
                # Enhanced single-scale processing
                input_tensor, original_size, new_size, padding = preprocess_image_microplastics(image_path)

                if torch.cuda.is_available():
                    input_tensor = input_tensor.cuda()

                with torch.no_grad():
                    d0, d1, d2, d3, d4, d5, d6 = model(input_tensor)
                    pred = d0[:, 0, :, :]

                mask = postprocess_mask_microplastics(pred, original_size, new_size, padding, image_path)
                cv2.imwrite(output_path, mask)

            print(f"Saved microplastic mask to {output_path}")
        except Exception as e:
            print(f"Error processing {image_file}: {str(e)}")

# Main function optimized for microplastics
def main():
    # Path to your trained U²-Net model
    model_path = "/content/u2net_model.pth"

    # Input and output directories
    input_dir = "/content/train"
    output_dir = "/content/train/train_microplastic_masks"
    refined_dir = "/content/train/microplastic_masks"

    # Load model
    print("Loading model for microplastic detection...")
    model = load_u2net(model_path)

    if model is None:
        print("Failed to load model. Exiting.")
        return

    # Generate masks with microplastic-optimized processing
    print("Generating microplastic masks...")
    process_directory_microplastics(model, input_dir, output_dir, use_multi_scale=True)

    # Enhanced refinement for microplastics
    print("Refining microplastic masks...")
    if not os.path.exists(refined_dir):
        os.makedirs(refined_dir)

    mask_files = [f for f in os.listdir(output_dir) if f.endswith('.png')]
    image_files = [f for f in os.listdir(input_dir)
                  if os.path.splitext(f)[1].lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']]

    for mask_file in mask_files:
        # The mask file now has the same name as the original image
        base_name = mask_file.replace('.png', '')
        matching_images = [f for f in image_files if base_name in f]

        if matching_images:
            original_image = matching_images[0]
            mask_path = os.path.join(output_dir, mask_file)
            original_path = os.path.join(input_dir, original_image)
            # Use the same filename for refined mask
            refined_path = os.path.join(refined_dir, f"{mask_file}")

            try:
                refine_microplastic_mask(mask_path, original_path, refined_path)
                print(f"Refined {mask_file}")
            except Exception as e:
                print(f"Error refining {mask_file}: {str(e)}")

    print("Microplastic detection process completed!")

if __name__ == "__main__":
    main()

Loading model for microplastic detection...
Model loaded successfully from /content/u2net_model.pth
Generating microplastic masks...
Processing Whole_303_jpg.rf.03e2066f112f3caa099dabcfadce6ea6.jpg for microplastics...
Saved microplastic mask to /content/train/train_microplastic_masks/Whole_303_jpg.rf.03e2066f112f3caa099dabcfadce6ea6.png
Processing Whole_068_jpg.rf.373a3100054ce436ac606c92c9a35ed0.jpg for microplastics...
Saved microplastic mask to /content/train/train_microplastic_masks/Whole_068_jpg.rf.373a3100054ce436ac606c92c9a35ed0.png
Processing Mix_071_jpg.rf.6e14005ba13305a3e550ea9c57477676.jpg for microplastics...
Saved microplastic mask to /content/train/train_microplastic_masks/Mix_071_jpg.rf.6e14005ba13305a3e550ea9c57477676.png
Processing Whole_206_jpg.rf.0046c89c548090f6a5f1aa1a93876f90.jpg for microplastics...
Saved microplastic mask to /content/train/train_microplastic_masks/Whole_206_jpg.rf.0046c89c548090f6a5f1aa1a93876f90.png
Processing Whole_241_jpg.rf.7fab01f6b6a76d3

In [ ]:
# import google.colab
# import zipfile
# import os

# def download_folder(folder_path, zip_name=None):
#     """
#     Compress and download a folder

#     Args:
#         folder_path: Path to the folder to download
#         zip_name: Name of the zip file (optional)
#     """
#     if zip_name is None:
#         zip_name = os.path.basename(folder_path) + '.zip'

#     # Create zip file
#     with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
#         for root, dirs, files_in_folder in os.walk(folder_path): # Renamed 'files' to 'files_in_folder'
#             for file in files_in_folder: # Used renamed variable
#                 file_path = os.path.join(root, file)
#                 # Add file to zip with relative path
#                 arcname = os.path.relpath(file_path, os.path.dirname(folder_path))
#                 zipf.write(file_path, arcname)

#     # Download the zip file
#     google.colab.files.download(zip_name) # Explicitly call download from google.colab.files

#     # Optional: Remove the zip file after download
#     # os.remove(zip_name)

# # Usage
# download_folder('/content/output1/microplastic_masks')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Copy folder to Drive (much faster than downloading)
source_folder = '/content/output1/microplastic_maskss'
destination_folder = '/content/drive/MyDrive/train_microplastic_masks'

# Remove existing if any
!rm -rf {destination_folder}

# Copy using system command (faster than shutil)
!cp -r {source_folder} {destination_folder}

print("Folder copied to Google Drive!")
print(f"Location: {destination_folder}")

# Optional: Unmount drive
# drive.flush_and_unmount()

Mounted at /content/drive
cp: cannot stat '/content/output1/microplastic_maskss': No such file or directory
Folder copied to Google Drive!
Location: /content/drive/MyDrive/train_microplastic_masks
